In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from itertools import product
from itertools import combinations as comb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from scipy.stats import rankdata

In [2]:
train = pd.read_csv('../data/train.csv')

test  = pd.read_csv('../data/test.csv')

In [3]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


In [4]:
test_ids = test['id'].copy()
train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

In [5]:
train['loan_paid_back'] = train['loan_paid_back'].astype(int)
X = train.drop(columns=['loan_paid_back'])
y = train['loan_paid_back']

In [6]:
grade_order = [
    'A1','A2','A3','A4','A5',
    'B1','B2','B3','B4','B5',
    'C1','C2','C3','C4','C5',
    'D1','D2','D3','D4','D5',
    'E1','E2','E3','E4','E5',
    'F1','F2','F3','F4','F5'
]
grade_map = {g: i+1 for i, g in enumerate(grade_order)}
X['grade_subgrade']    = X['grade_subgrade'].map(grade_map)
test['grade_subgrade'] = test['grade_subgrade'].map(grade_map)

In [7]:
cat_cols = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
for col in cat_cols:
    X[col]    = X[col].astype('category')
    test[col] = test[col].astype('category')


In [8]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
model = lgb.LGBMClassifier(
    objective='binary', metric='auc',
    n_estimators=1000, learning_rate=0.05,
    num_leaves=31, random_state=42,
    n_jobs=-1, verbose=-1
)
scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
print(f'LightGBM CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')


LightGBM CV AUC: 0.9224 ± 0.0012


In [9]:
model.fit(X, y)
final_pred = model.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'id': test_ids,
    'loan_paid_back': final_pred
})
submission.to_csv('../submissions/submission_simple.csv', index=False)
print(f'Rows: {len(submission)}')
print(submission.head())

Rows: 254569
       id  loan_paid_back
0  593994        0.926983
1  593995        0.981172
2  593996        0.562164
3  593997        0.925394
4  593998        0.971894
